# The Illusion of "memory"

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

In [8]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')

# Gemini (this course also uses GOOGLE_API_KEY)
if not gemini_api_key:
    print("GEMINI_API_KEY is missing — check the name in .env")
elif not gemini_api_key.startswith(("AIz", "AQ.")):
    print("GEMINI_API_KEY is set, but does not start with AIz or AQ.")
else:
    print(f"GEMINI_API_KEY loaded, starts with {gemini_api_key[:4]}")

GEMINI_API_KEY loaded, starts with AIza


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [10]:
from openai import OpenAI
gemini = OpenAI(base_url='https://generativelanguage.googleapis.com/v1beta/openai/', api_key= gemini_api_key)

### A message to OpenAI is a list of dicts

In [11]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
]

In [12]:
response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
response.choices[0].message.content

"Hi Ed! It's great to meet you. How are you doing today? Is there anything I can help you with?"

### OK let's now ask a follow-up question

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [15]:
response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
response.choices[0].message.content

"I don’t know your name. As an AI, I don’t have access to your personal identity or private information unless you share it with me during our conversation. \n\nIf you’d like, you can tell me what you'd like to be called!"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [16]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [17]:
response = gemini.chat.completions.create(model="gemini-3.1-flash-lite", messages=messages)
response.choices[0].message.content

'Your name is Ed!'